In [ ]:
import os
from dotenv import load_dotenv
from mwrogue.esports_client import EsportsClient
from mwrogue.auth_credentials import AuthCredentials

load_dotenv()

credentials = AuthCredentials(
    username=os.getenv("LEAGUEPEDIA_USER"),
    password=os.getenv("LEAGUEPEDIA_PASS"),
)
site = EsportsClient("lol", credentials=credentials)

In [ ]:
print(load_dotenv())                   # True means the .env file was found
#print(os.getenv("LEAGUEPEDIA_USER"))   # should show YourName@yourbot

info = site.client.api("query", meta="userinfo", uiprop="groups")
print("anon" not in info["query"]["userinfo"])   # True = logged in

In [ ]:
import duckdb
import time

page = "LCK/2026 Season/Rounds 1-2"

con = duckdb.connect("lol_rankings.duckdb")
con.execute("DROP TABLE IF EXISTS raw.postgame_json_metadata")   # delete this line after one run
con.execute(open("docs/raw_schema.sql").read())

In [ ]:
import html

def clean(v):
    return None if v in (None, "") else html.unescape(v)

In [ ]:
def pull_table(cargo_table, raw_table, cols, page):
    where = f"OverviewPage = '{page}'"

    rows = site.cargo_client.query(tables=cargo_table, fields=", ".join(cols), where=where)

    values = [[clean(r.get(c, r.get(c.replace("_", " ")))) for c in cols] for r in rows]

    col_names = ", ".join(cols)
    placeholders = ", ".join("?" for _ in cols)

    if not values:
        raise ValueError(f"No rows for {cargo_table} / {page}")

    con.execute("BEGIN")
    con.execute(f"DELETE FROM raw.{raw_table} WHERE OverviewPage = ?", [page])
    con.executemany(f"INSERT INTO raw.{raw_table} ({col_names}) VALUES ({placeholders})", values)
    con.execute("COMMIT")

    time.sleep(5)
    return len(values)

In [ ]:
games_cols = ["GameId", "MatchId", "OverviewPage", "Team1", "Team2", "WinTeam", "DateTime_UTC"]
print(pull_table("ScoreboardGames", "scoreboard_games", games_cols, page))

In [ ]:
print(con.execute("""
    SELECT COUNT(*), COUNT(DISTINCT GameId)
    FROM raw.scoreboard_games
    WHERE OverviewPage = ?
""", [page]).fetchone())

In [ ]:
teams_cols = ["GameTeamId", "GameId", "MatchId", "OverviewPage", "Team", "Side",
              "IsWinner", "Dragons", "Barons", "Towers", "Gold", "Kills"]

print(pull_table("ScoreboardTeams", "scoreboard_teams", teams_cols, page))

In [ ]:
print(con.execute("""
    SELECT COUNT(*) FROM raw.scoreboard_teams
    WHERE OverviewPage = ? AND IsWinner
""", [page]).fetchone())

In [ ]:
players_cols = ["GameRoleId", "GameId", "GameTeamId", "MatchId", "OverviewPage", "Link", "Name",
                "Team", "Role", "Side", "Champion", "Kills", "Deaths", "Assists", "Gold", "CS",
                "DamageToChampions", "VisionScore", "DateTime_UTC"]

print(pull_table("ScoreboardPlayers", "scoreboard_players", players_cols, page))

In [ ]:
print(con.execute("""
    SELECT COUNT(*)
    FROM raw.scoreboard_players p
    JOIN raw.scoreboard_teams t USING (GameTeamId)
    JOIN raw.scoreboard_games g ON t.GameId = g.GameId
    WHERE p.OverviewPage = ?
""", [page]).fetchone())

In [ ]:
match_cols = ["MatchId", "OverviewPage", "Team1", "Team2", "Winner", "Team1Score", "Team2Score",
              "FF", "IsNullified", "BestOf", "DateTime_UTC", "Tab", "Patch"]

print(pull_table("MatchSchedule", "match_schedule", match_cols, page))

In [ ]:
print(con.execute("""
    SELECT SUM(Team1Score + Team2Score) FROM raw.match_schedule WHERE OverviewPage = ?
""", [page]).fetchone())

In [ ]:
con.close()